# 🍽️ Assignment 4 — Restaurant Expert Chatbot
### Fully Automated Version — Built on Assignment 3 ABSA Pipeline
**Run cells one by one from top to bottom.**

In [1]:
#  Install all dependencies
!pip install contractions imbalanced-learn scikit-learn spacy transformers torch nltk --quiet
!python -m spacy download en_core_web_sm --quiet
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
print('✅ All dependencies installed')

zsh:1: command not found: pip
zsh:1: command not found: python
✅ All dependencies installed


In [ ]:
# Connect to Google Drive and find data files
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/absa')
else:
    PROJECT_ROOT = Path.cwd()

TRAIN_XML = PROJECT_ROOT / 'data' / 'raw' / 'Restaurants_Train_v2.xml'
TEST_XML  = PROJECT_ROOT / 'data' / 'raw' / 'Restaurants_Test_Gold.xml'

print(f'Train XML exists: {TRAIN_XML.exists()}')
print(f'Test XML  exists: {TEST_XML.exists()}')

if not TRAIN_XML.exists():
    print('\n❌ ERROR: Data files not found!')
    print('Make sure your absa folder is at: MyDrive/absa/data/raw/')

Mounted at /content/drive
Train XML exists: True
Test XML  exists: True


In [ ]:
# Load all imports
import os, re, xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import contractions
import spacy
from dataclasses import dataclass
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from transformers import pipeline as hf_pipeline
from nltk.corpus import stopwords

# Load spaCy English model
nlp = spacy.load('en_core_web_sm')

# Load NLTK stopwords — automated, no manual list needed
STOPWORDS = set(stopwords.words('english'))

TOKEN_RE       = re.compile(r'[A-Za-z][A-Za-z\-\']+')
POLARITY_ORDER = ['positive', 'negative', 'neutral', 'conflict']
CATEGORIES     = ['food', 'service', 'price', 'ambience', 'miscellaneous']

print('✅ All imports loaded')
print(f'   NLTK stopwords loaded: {len(STOPWORDS)} words')

✅ All imports loaded
   NLTK stopwords loaded: 198 words


In [ ]:

# Text Cleaning Functions


def normalize_text(text):
    """Fix extra spaces in any text"""
    return ' '.join((text or '').split())

def normalize_term(term):
    """Clean up aspect words — lowercase, remove special characters"""
    term = normalize_text(term).lower().strip()
    term = re.sub(r'[^a-z0-9\s\-\']', ' ', term)
    return re.sub(r'\s+', ' ', term)

def clean_text(text):
    """Full cleaning for review sentences — expand contractions, remove links"""
    text = contractions.fix(str(text)).lower()
    text = re.sub(r'http\S+|www\S+|<.*?>', '', text)
    return re.sub(r'\s+', ' ', text).strip()

print('✅ Text cleaning functions ready')

✅ Text cleaning functions ready


In [ ]:

# XML Parser


@dataclass(frozen=True)
class ParsedDataset:
    name: str
    sentences: pd.DataFrame
    aspects: pd.DataFrame
    categories: pd.DataFrame

def parse_restaurant_xml(path, split_name):
    """Reads the XML file and converts it into 3 neat tables"""
    root = ET.parse(path).getroot()
    sentences, aspects, categories = [], [], []

    for sentence in root.findall('.//sentence'):
        sid     = sentence.attrib['id']
        text    = normalize_text(sentence.findtext('text', default=''))
        at_node = sentence.find('aspectTerms')
        ac_node = sentence.find('aspectCategories')
        at_list = at_node.findall('aspectTerm')    if at_node is not None else []
        ac_list = ac_node.findall('aspectCategory') if ac_node is not None else []

        sentences.append({
            'split': split_name, 'sentence_id': sid, 'text': text,
            'token_count': len(TOKEN_RE.findall(text)),
            'aspect_term_count': len(at_list),
            'aspect_category_count': len(ac_list)
        })
        for idx, asp in enumerate(at_list):
            aspects.append({
                'split': split_name, 'sentence_id': sid,
                'aspect_id': f'{sid}::term::{idx}', 'text': text,
                'term': asp.attrib.get('term', ''),
                'term_normalized': normalize_term(asp.attrib.get('term', '')),
                'polarity': asp.attrib.get('polarity', '').lower()
            })
        for idx, cat in enumerate(ac_list):
            categories.append({
                'split': split_name, 'sentence_id': sid,
                'category_id': f'{sid}::cat::{idx}', 'text': text,
                'category': cat.attrib.get('category', '').lower(),
                'polarity': cat.attrib.get('polarity', '').lower()
            })

    return ParsedDataset(
        name=split_name,
        sentences=pd.DataFrame(sentences),
        aspects=pd.DataFrame(aspects),
        categories=pd.DataFrame(categories)
    )

print('✅ XML parser ready')

✅ XML parser ready


In [ ]:
# Automated Aspect Extraction using spaCy

NON_ASPECTS = set([
    'good','great','bad','excellent','amazing','terrible',
    'delicious','friendly','nice','love','loved','horrible',
    'poor','slow','wonderful','best','worst','perfect','awful',
    'unhelpful','overcooked','undercooked','rude','cold','hot',
    'loud','dirty','clean','fresh','stale','burnt','raw'
])

# Still keep the lexicon as backup for known restaurant words
def build_extraction_lexicon(df):
    """Learn known aspect words from training data"""
    tc = Counter(df['term_normalized'])
    hc = Counter(df['term_normalized'].apply(lambda t: t.split()[-1]))
    return (
        {t for t,c in tc.items() if c>=2 and len(t.split())==1},
        {t for t,c in tc.items() if c>=2 and 1<len(t.split())<=3},
        {h for h,c in hc.items() if c>=3 and h not in STOPWORDS}
    )

def extract_aspects_spacy(text, single, multi, heads):
    """
    AUTOMATED aspect extraction using spaCy.
    spaCy reads the sentence and finds nouns automatically.
    Also checks against the learned lexicon from training data.
    """
    cleaned = clean_text(text)
    doc     = nlp(cleaned)
    found   = set()

    # METHOD 1: spaCy finds nouns automatically
    # POS = Part of Speech tagging
    # NOUN = a noun word,  PROPN = a proper noun (like a name)
    for token in doc:
        word = token.text.lower()
        if (token.pos_ in ('NOUN', 'PROPN')           # is it a noun?
                and word not in STOPWORDS              # not a stopword?
                and word not in NON_ASPECTS            # not a feeling word?
                and len(word) > 2):                    # not too short?
            found.add(word)

    # METHOD 2: also check noun chunks (multi-word nouns)
    # e.g. "wait staff", "dining room", "chicken wings"
    for chunk in doc.noun_chunks:
        phrase = chunk.text.lower().strip()
        if (phrase not in STOPWORDS
                and phrase not in NON_ASPECTS
                and len(phrase) > 2):
            found.add(phrase)

    # METHOD 3: also check against the trained lexicon as backup
    tokens = re.findall(r'[a-z][a-z\-\']+', cleaned)
    for size in [3, 2]:
        for i in range(len(tokens)-size+1):
            p = ' '.join(tokens[i:i+size])
            if p in multi: found.add(p)
    for t in tokens:
        if t in single: found.add(t)

    return list(found)

print('✅ Automated aspect extraction ready (spaCy + lexicon)')

✅ Automated aspect extraction ready (spaCy + lexicon)


In [ ]:
# Automated Category Mapping using BART

print('Loading BART zero-shot classifier...')
print('(This takes about 1-2 minutes the first time)')

zero_shot = hf_pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli'
)

def predict_category(term):
    """
    AUTOMATED category prediction using BART.
    No manual keyword lists needed.
    BART reads the word and decides which category it belongs to.
    """
    try:
        result = zero_shot(term, CATEGORIES)
        return result['labels'][0]   # top predicted category
    except Exception:
        # Fallback — if BART fails for any reason, use simple keyword match
        term_lower = term.lower()
        if any(w in term_lower for w in ['food','dish','pasta','pizza','taste','flavor']): return 'food'
        if any(w in term_lower for w in ['staff','waiter','service','server']): return 'service'
        if any(w in term_lower for w in ['price','bill','cost','expensive','cheap']): return 'price'
        if any(w in term_lower for w in ['atmosphere','decor','music','ambience']): return 'ambience'
        return 'miscellaneous'

# Test it works
print('\nTesting BART category predictions:')
test_words = ['pasta', 'waiter', 'bill', 'atmosphere', 'reservation']
for w in test_words:
    print(f'   "{w}" → {predict_category(w)}')
print('\n✅ BART category mapper ready')

Loading BART zero-shot classifier...
(This takes about 1-2 minutes the first time)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


Testing BART category predictions:
   "pasta" → food
   "waiter" → service
   "bill" → price
   "atmosphere" → ambience
   "reservation" → ambience

✅ BART category mapper ready


In [ ]:
# Sentiment Model


def make_feature(text, term):
    """
    Wraps the aspect word in special tags.
    Tells the model which word to focus on.
    Example: 'the [ASPECT] pasta [/ASPECT] was cold'
    """
    txt = clean_text(text)
    tn  = normalize_term(term)
    idx = txt.find(tn)
    if idx == -1:
        return f'[ASPECT] {tn} [/ASPECT] || {txt}'
    return txt[:idx] + f' [ASPECT] {txt[idx:idx+len(tn)]} [/ASPECT] ' + txt[idx+len(tn):]

print('✅ Feature engineering function ready')

✅ Feature engineering function ready


In [ ]:
# Intent Detection

GREETINGS  = {'hi','hello','hey','howdy','greetings','morning','evening'}
FAREWELL   = {'bye','goodbye','quit','thanks','thank'}
HELP_WORDS = {'help','capabilities'}

def detect_intent(text):
    tokens = set(re.findall(r'[a-z]+', text.lower()))

    if tokens & GREETINGS: return 'greeting'
    if tokens & FAREWELL:  return 'farewell'
    if tokens & HELP_WORDS:return 'help'

    doc   = nlp(clean_text(text))
    nouns = [token.text.lower() for token in doc
             if token.pos_ in ('NOUN', 'PROPN')
             and len(token.text) > 2]

    if nouns:
        try:
            result = zero_shot(
                text,
                ['restaurant review', 'general conversation']
            )
            if result['labels'][0] == 'restaurant review':
                return 'restaurant_query'
        except Exception:
            basic = {
                'food','meal','dish','menu','taste','service','staff',
                'waiter','price','bill','cost','ambience','atmosphere',
                'restaurant','table','reservation','drink','wine'
            }
            if tokens & basic:
                return 'restaurant_query'

    return 'general'


EMOJI = {'positive':'😊', 'negative':'😞', 'neutral':'😐', 'conflict':'🤔'}

def format_absa_response(results):
    if not results:
        return ('I could not identify specific aspects in your message.\n'
                'Try mentioning food, service, price, or ambience specifically.')
    lines  = ['Here is what I found:\n']
    by_cat = defaultdict(list)
    for a in results:
        by_cat[a['category']].append(a)
    for cat, items in by_cat.items():
        dominant = Counter(i['sentiment'] for i in items).most_common(1)[0][0]
        # ← UPDATED: removes "the", "a", "an" and deduplicates
        unique_terms = set()
        for i in items:
            aspect = re.sub(r'^(the|a|an)\s+', '', i['aspect'].strip())
            unique_terms.add(aspect)
        terms = ', '.join(unique_terms)
        lines.append(f'  {EMOJI.get(dominant,"")} {cat.upper()}: {dominant}  (about: {terms})')
    lines.append('\nWant to ask about a specific aspect like food quality or service?')
    return '\n'.join(lines)

def general_responses(intent):
    if intent == 'greeting':
        return ('Hello! 👋 I am your restaurant review expert chatbot.\n'
                'I can analyse reviews and tell you how people feel about\n'
                'the food, service, price, or ambience.\n'
                'Just type a review or ask a question!')
    if intent == 'farewell':
        return 'Thanks for chatting! Hope the insights were helpful. Goodbye! 👋'
    if intent == 'help':
        return ('I can help you with:\n'
                '  • Analysing sentiment in restaurant reviews\n'
                '  • Identifying which aspects are positive or negative\n'
                '  • Answering questions about food, service, price, ambience\n\n'
                'Try typing:\n'
                '  "The pasta was cold but the waiter was friendly"\n'
                '  "What do people think about the service?"')
    # ← UPDATED: better fallback message
    return ('I am not sure I understood that.\n'
            'Try typing a restaurant review like:\n'
            '  "The pasta was cold but the waiter was friendly"')

print('✅ Intent detection and response functions ready')

✅ Intent detection and response functions ready


In [ ]:
# Train the chatbot


print('🔧 Step 1/4: Loading XML data...')
train_data = parse_restaurant_xml(TRAIN_XML, 'train')
print(f'   Loaded {len(train_data.aspects)} training aspect annotations')

print('🔧 Step 2/4: Building aspect extraction lexicon...')
single_lex, multi_lex, head_lex = build_extraction_lexicon(train_data.aspects)
print(f'   Single words: {len(single_lex)} | Multi-word: {len(multi_lex)} | Head words: {len(head_lex)}')

print('🔧 Step 3/4: Preparing sentiment training data...')
train_df              = train_data.aspects.copy()
train_df['clean_text']= train_df['text'].apply(clean_text)
train_df['feature']   = train_df.apply(
    lambda r: make_feature(r['text'], r['term_normalized']), axis=1
)

print('🔧 Step 4/4: Training TF-IDF + SMOTE + Logistic Regression...')
tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=5000, sublinear_tf=True)
X_vec = tfidf.fit_transform(train_df['feature'])
y_all = train_df['polarity']

smote    = SMOTE(random_state=42)
X_s, y_s = smote.fit_resample(X_vec, y_all)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_s, y_s)

print('\n✅ Chatbot fully trained and ready!')
print(f'   Sentiment classes: {list(clf.classes_)}')

🔧 Step 1/4: Loading XML data...
   Loaded 3693 training aspect annotations
🔧 Step 2/4: Building aspect extraction lexicon...
   Single words: 215 | Multi-word: 85 | Head words: 203
🔧 Step 3/4: Preparing sentiment training data...
🔧 Step 4/4: Training TF-IDF + SMOTE + Logistic Regression...

✅ Chatbot fully trained and ready!
   Sentiment classes: ['conflict', 'negative', 'neutral', 'positive']


In [ ]:

# The main chat function

def analyse(text):
    aspects = extract_aspects_spacy(text, single_lex, multi_lex, head_lex)
    results = []
    seen    = set()  # track already processed aspects

    for asp in aspects:
        # Remove "the", "a", "an" from the start
        clean_asp = re.sub(r'^(the|a|an)\s+', '', asp.strip())

        # Skip if we already processed this aspect
        if clean_asp in seen:
            continue
        seen.add(clean_asp)

        feat = make_feature(text, clean_asp)
        vec  = tfidf.transform([feat])
        results.append({
            'aspect':    clean_asp,
            'category':  predict_category(clean_asp),
            'sentiment': clf.predict(vec)[0]
        })
    return results

def chat(user_input):
    """
    Main chatbot function.
    Takes user message → returns chatbot reply.
    """
    if not user_input.strip():
        return 'Please type something!'
    intent = detect_intent(user_input)
    if intent == 'restaurant_query':
        return format_absa_response(analyse(user_input))
    return general_responses(intent)

print('✅ chat() function ready')

✅ chat() function ready


In [ ]:
#  DEMO

demo_inputs = [
    'Hello!',
    'What can you help me with?',
    'The pasta was cold but the waiter was incredibly friendly and fast.',
    'Overpriced for the tiny portions, though the atmosphere was cozy.',
    'Great value and quick service, but the music was too loud to talk.',
    'The staff was rude and the waiting time was too long.',
    'Amazing desserts and the ambience was perfect for a date night.',
    'The bill was outrageous for such a small meal.',
    'The risotto was overcooked and the sommelier was rude.',
    'Goodbye!'
]
print('=' * 60)
print('  🍽️  RESTAURANT EXPERT CHATBOT — VIVA DEMO')
print('=' * 60)

for q in demo_inputs:
    print(f'\n👤 You: {q}')
    print(f'🤖 Bot: {chat(q)}')
    print('-' * 50)

  🍽️  RESTAURANT EXPERT CHATBOT — VIVA DEMO

👤 You: Hello!
🤖 Bot: Hello! 👋 I am your restaurant review expert chatbot.
I can analyse reviews and tell you how people feel about
the food, service, price, or ambience.
Just type a review or ask a question!
--------------------------------------------------

👤 You: What can you help me with?
🤖 Bot: I can help you with:
  • Analysing sentiment in restaurant reviews
  • Identifying which aspects are positive or negative
  • Answering questions about food, service, price, ambience

Try typing:
  "The pasta was cold but the waiter was friendly"
  "What do people think about the service?"
--------------------------------------------------

👤 You: The pasta was cold but the waiter was incredibly friendly and fast.
🤖 Bot: Here is what I found:

  😞 FOOD: negative  (about: pasta)
  😞 SERVICE: negative  (about: waiter)

Want to ask about a specific aspect like food quality or service?
--------------------------------------------------

👤 You: Overpr

In [ ]:
# INTERACTIVE MODE (for examiner questions)
# Type STOP to end

print('🍽️  Interactive mode — type your message below.')
print('Type STOP to end.\n')

while True:
    user_input = input('👤 You: ').strip()
    if user_input.upper() == 'STOP':
        print('🤖 Bot: Goodbye! 👋')
        break
    if user_input:
        print(f'🤖 Bot: {chat(user_input)}\n')

🍽️  Interactive mode — type your message below.
Type STOP to end.

👤 You: The spagetti was so good
🤖 Bot: Here is what I found:

  😊 FOOD: positive  (about: spagetti)

Want to ask about a specific aspect like food quality or service?

👤 You: That goal was out of world
🤖 Bot: I am not sure I understood that.
Try typing a restaurant review like:
  "The pasta was cold but the waiter was friendly"

👤 You: stop
🤖 Bot: Goodbye! 👋
